In [1]:
import numpy as np

from pprint import pprint

import torch

import re
import unicodedata
from collections import Counter

from typing import Literal

import torch.nn as nn
import torch.nn.functional as F

In [9]:
with open('../data/TinyStories_train_100k.txt', 'r', encoding = 'utf-8') as f:
    text = f.read().lower()

In [3]:
print("length of dataset in characters: ", len(text))
print(text[:2000])

length of dataset in characters:  89400820
one day, a little girl named lily found a needle in her room. she knew it was difficult to play with it because it was sharp. lily wanted to share the needle with her mom, so she could sew a button on her shirt.
lily went to her mom and said, "mom, i found this needle. can you share it with me and sew my shirt?" her mom smiled and said, "yes, lily, we can share the needle and fix your shirt."
together, they shared the needle and sewed the button on lily's shirt. it was not difficult for them because they were sharing and helping each other. after they finished, lily thanked her mom for sharing the needle and fixing her shirt. they both felt happy because they had shared and worked together.
<|endoftext|>
once upon a time, there was a little car named beep. beep loved to go fast and play in the sun. beep was a healthy car because he always had good fuel. good fuel made beep happy and strong.
one day, beep was driving in the park when he saw a b

In [4]:
sep = "<|endoftext|>"
pprint(text[:2000].split(sep=sep))

['one day, a little girl named lily found a needle in her room. she knew it '
 'was difficult to play with it because it was sharp. lily wanted to share the '
 'needle with her mom, so she could sew a button on her shirt.\n'
 'lily went to her mom and said, "mom, i found this needle. can you share it '
 'with me and sew my shirt?" her mom smiled and said, "yes, lily, we can share '
 'the needle and fix your shirt."\n'
 "together, they shared the needle and sewed the button on lily's shirt. it "
 'was not difficult for them because they were sharing and helping each other. '
 'after they finished, lily thanked her mom for sharing the needle and fixing '
 'her shirt. they both felt happy because they had shared and worked '
 'together.\n',
 '\n'
 'once upon a time, there was a little car named beep. beep loved to go fast '
 'and play in the sun. beep was a healthy car because he always had good fuel. '
 'good fuel made beep happy and strong.\n'
 'one day, beep was driving in the park whe

In [5]:
sentence = text[:2000].split(sep=sep)[0]
sentence

'one day, a little girl named lily found a needle in her room. she knew it was difficult to play with it because it was sharp. lily wanted to share the needle with her mom, so she could sew a button on her shirt.\nlily went to her mom and said, "mom, i found this needle. can you share it with me and sew my shirt?" her mom smiled and said, "yes, lily, we can share the needle and fix your shirt."\ntogether, they shared the needle and sewed the button on lily\'s shirt. it was not difficult for them because they were sharing and helping each other. after they finished, lily thanked her mom for sharing the needle and fixing her shirt. they both felt happy because they had shared and worked together.\n'

In [6]:
word = sentence.split()[0]
word

'one'

In [7]:
characters = list(word)
characters

['o', 'n', 'e']

In [8]:
sep = "<|endoftext|>"
tokenized_stories = []
total_tokens = []

for chunk in text.split(sep=sep):
    sentence = list(chunk) + [sep]
    tokenized_stories.append(sentence)
    
    total_tokens.extend(list(chunk))
    total_tokens.append(sep)
tokenized_stories.pop()
total_tokens.pop()

tokenized_stories[0]

['o',
 'n',
 'e',
 ' ',
 'd',
 'a',
 'y',
 ',',
 ' ',
 'a',
 ' ',
 'l',
 'i',
 't',
 't',
 'l',
 'e',
 ' ',
 'g',
 'i',
 'r',
 'l',
 ' ',
 'n',
 'a',
 'm',
 'e',
 'd',
 ' ',
 'l',
 'i',
 'l',
 'y',
 ' ',
 'f',
 'o',
 'u',
 'n',
 'd',
 ' ',
 'a',
 ' ',
 'n',
 'e',
 'e',
 'd',
 'l',
 'e',
 ' ',
 'i',
 'n',
 ' ',
 'h',
 'e',
 'r',
 ' ',
 'r',
 'o',
 'o',
 'm',
 '.',
 ' ',
 's',
 'h',
 'e',
 ' ',
 'k',
 'n',
 'e',
 'w',
 ' ',
 'i',
 't',
 ' ',
 'w',
 'a',
 's',
 ' ',
 'd',
 'i',
 'f',
 'f',
 'i',
 'c',
 'u',
 'l',
 't',
 ' ',
 't',
 'o',
 ' ',
 'p',
 'l',
 'a',
 'y',
 ' ',
 'w',
 'i',
 't',
 'h',
 ' ',
 'i',
 't',
 ' ',
 'b',
 'e',
 'c',
 'a',
 'u',
 's',
 'e',
 ' ',
 'i',
 't',
 ' ',
 'w',
 'a',
 's',
 ' ',
 's',
 'h',
 'a',
 'r',
 'p',
 '.',
 ' ',
 'l',
 'i',
 'l',
 'y',
 ' ',
 'w',
 'a',
 'n',
 't',
 'e',
 'd',
 ' ',
 't',
 'o',
 ' ',
 's',
 'h',
 'a',
 'r',
 'e',
 ' ',
 't',
 'h',
 'e',
 ' ',
 'n',
 'e',
 'e',
 'd',
 'l',
 'e',
 ' ',
 'w',
 'i',
 't',
 'h',
 ' ',
 'h',
 'e',
 'r',
 ' '

In [9]:
stories = len(tokenized_stories)
avg_story_length = np.mean([len(story) for story in tokenized_stories])
print(f"Total number of charcters in entire dataset: {len(total_tokens)}")
print(f"Total number of stories: {stories}, Average number of characters per story: {avg_story_length: .2f}")

Total number of charcters in entire dataset: 88200820
Total number of stories: 100000, Average number of characters per story:  882.01


In [10]:
len(tokenized_stories[0])

701

In [11]:
vocab = sorted(set(token for token in total_tokens))
print(f"Length of Vocabulary: {len(vocab)}")
print(vocab)

Length of Vocabulary: 83
['\t', '\n', ' ', '!', '"', '#', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<|endoftext|>', '?', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '\xa0', '¡', '«', '´', '»', 'á', 'é', 'í', 'ñ', 'ó', 'ö', '\u2009', '\u200a', '\u200b', '–', '—', '―', '‘', '’', '“', '”', '…']


In [12]:
sep = "<|endoftext|>"

text_clean = unicodedata.normalize("NFKC", text).lower()

replace_map = str.maketrans({
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
    "–": "-",
    "—": "-",
    "―": "-",
    "…": "...",
    "\xa0": " ",   # NBSP
    "\u2009": " ", # thin space
    "\u200a": " ", # hair space
    "\u200b": "",  # zero-width space
})

text_clean = text_clean.translate(replace_map)

allowed = set("abcdefghijklmnopqrstuvwxyz0123456789 .,!?;:'\"-()/\n\t")

stories = []
for chunk in text_clean.split(sep=sep):
    chunk = "".join(ch for ch in chunk if ch in allowed)
    chunk = re.sub(r"[ ]{2,}", " ", chunk)  # collapse repeated spaces
    chunk = chunk.strip()
    if chunk:
        stories.append(chunk)
        
text_clean = f"{sep}".join(stories) + sep

tokenized_stories = [list(s) + [sep] for s in stories]
total_tokens = [ch for s in stories for ch in s] + [sep] * len(stories)

In [13]:
stories[0]

'one day, a little girl named lily found a needle in her room. she knew it was difficult to play with it because it was sharp. lily wanted to share the needle with her mom, so she could sew a button on her shirt.\nlily went to her mom and said, "mom, i found this needle. can you share it with me and sew my shirt?" her mom smiled and said, "yes, lily, we can share the needle and fix your shirt."\ntogether, they shared the needle and sewed the button on lily\'s shirt. it was not difficult for them because they were sharing and helping each other. after they finished, lily thanked her mom for sharing the needle and fixing her shirt. they both felt happy because they had shared and worked together.'

In [14]:
tokenized_stories[0]

['o',
 'n',
 'e',
 ' ',
 'd',
 'a',
 'y',
 ',',
 ' ',
 'a',
 ' ',
 'l',
 'i',
 't',
 't',
 'l',
 'e',
 ' ',
 'g',
 'i',
 'r',
 'l',
 ' ',
 'n',
 'a',
 'm',
 'e',
 'd',
 ' ',
 'l',
 'i',
 'l',
 'y',
 ' ',
 'f',
 'o',
 'u',
 'n',
 'd',
 ' ',
 'a',
 ' ',
 'n',
 'e',
 'e',
 'd',
 'l',
 'e',
 ' ',
 'i',
 'n',
 ' ',
 'h',
 'e',
 'r',
 ' ',
 'r',
 'o',
 'o',
 'm',
 '.',
 ' ',
 's',
 'h',
 'e',
 ' ',
 'k',
 'n',
 'e',
 'w',
 ' ',
 'i',
 't',
 ' ',
 'w',
 'a',
 's',
 ' ',
 'd',
 'i',
 'f',
 'f',
 'i',
 'c',
 'u',
 'l',
 't',
 ' ',
 't',
 'o',
 ' ',
 'p',
 'l',
 'a',
 'y',
 ' ',
 'w',
 'i',
 't',
 'h',
 ' ',
 'i',
 't',
 ' ',
 'b',
 'e',
 'c',
 'a',
 'u',
 's',
 'e',
 ' ',
 'i',
 't',
 ' ',
 'w',
 'a',
 's',
 ' ',
 's',
 'h',
 'a',
 'r',
 'p',
 '.',
 ' ',
 'l',
 'i',
 'l',
 'y',
 ' ',
 'w',
 'a',
 'n',
 't',
 'e',
 'd',
 ' ',
 't',
 'o',
 ' ',
 's',
 'h',
 'a',
 'r',
 'e',
 ' ',
 't',
 'h',
 'e',
 ' ',
 'n',
 'e',
 'e',
 'd',
 'l',
 'e',
 ' ',
 'w',
 'i',
 't',
 'h',
 ' ',
 'h',
 'e',
 'r',
 ' '

In [15]:
pprint(text_clean[:750])

('one day, a little girl named lily found a needle in her room. she knew it '
 'was difficult to play with it because it was sharp. lily wanted to share the '
 'needle with her mom, so she could sew a button on her shirt.\n'
 'lily went to her mom and said, "mom, i found this needle. can you share it '
 'with me and sew my shirt?" her mom smiled and said, "yes, lily, we can share '
 'the needle and fix your shirt."\n'
 "together, they shared the needle and sewed the button on lily's shirt. it "
 'was not difficult for them because they were sharing and helping each other. '
 'after they finished, lily thanked her mom for sharing the needle and fixing '
 'her shirt. they both felt happy because they had shared and worked '
 'together.<|endoftext|>once upon a time, there was a little c')


In [16]:
pprint(text_clean[-100:])

('the little girl smiled and hugged her mom back. she was so proud of her '
 'accomplishment.<|endoftext|>')


In [17]:
num_stories = len(tokenized_stories)
avg_story_length = np.mean([len(story) for story in tokenized_stories])
print(f"Total number of characters in entire dataset: {len(total_tokens)}")
print(f"Total number of stories: {num_stories}, Average number of characters per story: {avg_story_length: .2f}")

Total number of characters in entire dataset: 87936505
Total number of stories: 99983, Average number of characters per story:  879.51


In [18]:
vocab = sorted(set(total_tokens))
print(f"Length of Vocabulary: {len(vocab)}")
print(vocab)

Length of Vocabulary: 52
['\t', '\n', ' ', '!', '"', "'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<|endoftext|>', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [19]:
stoi = {ch:i for i, ch in enumerate(vocab)}
itos = {i:ch for i, ch in enumerate(vocab)}
print(stoi)
print(itos)

{'\t': 0, '\n': 1, ' ': 2, '!': 3, '"': 4, "'": 5, '(': 6, ')': 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '<|endoftext|>': 24, '?': 25, 'a': 26, 'b': 27, 'c': 28, 'd': 29, 'e': 30, 'f': 31, 'g': 32, 'h': 33, 'i': 34, 'j': 35, 'k': 36, 'l': 37, 'm': 38, 'n': 39, 'o': 40, 'p': 41, 'q': 42, 'r': 43, 's': 44, 't': 45, 'u': 46, 'v': 47, 'w': 48, 'x': 49, 'y': 50, 'z': 51}
{0: '\t', 1: '\n', 2: ' ', 3: '!', 4: '"', 5: "'", 6: '(', 7: ')', 8: ',', 9: '-', 10: '.', 11: '/', 12: '0', 13: '1', 14: '2', 15: '3', 16: '4', 17: '5', 18: '6', 19: '7', 20: '8', 21: '9', 22: ':', 23: ';', 24: '<|endoftext|>', 25: '?', 26: 'a', 27: 'b', 28: 'c', 29: 'd', 30: 'e', 31: 'f', 32: 'g', 33: 'h', 34: 'i', 35: 'j', 36: 'k', 37: 'l', 38: 'm', 39: 'n', 40: 'o', 41: 'p', 42: 'q', 43: 'r', 44: 's', 45: 't', 46: 'u', 47: 'v', 48: 'w', 49: 'x', 50: 'y', 51: 'z'}


In [20]:
decode = lambda e: ''.join([itos[n] for n in e])     # Convert a list of numbers of to story

def encode(text: str) -> list:  # Convert a story to a list of numbers
    num_list = []
    c = 0
    while c < len(text):
        if text.startswith(sep, c):
            num_list.append(stoi[sep])
            c += len(sep) - 1
        else:
            num_list.append(stoi[text[c]])
        c += 1
    return num_list

print(encode("hello world!"))
print(decode(encode("hello world!")))

[33, 30, 37, 37, 40, 2, 48, 40, 43, 37, 29, 3]
hello world!


In [21]:
len(sep)

13

In [22]:
num = 0
c = 0
while c < len(text_clean[:750]):
    if text_clean[c] == '<':
        num += 1
    c += 1
print(f"Total number of occurences of {sep}: {num}")

Total number of occurences of <|endoftext|>: 1


In [23]:
print(encode(text_clean[:750]))
print(decode(encode(text_clean[:750])))
print(f"Length of original string: {len(text_clean[:750]) - num*(len(sep) - 1)}")
print(f"Length of converted number list: {len(encode(text_clean[:750]))}")

[40, 39, 30, 2, 29, 26, 50, 8, 2, 26, 2, 37, 34, 45, 45, 37, 30, 2, 32, 34, 43, 37, 2, 39, 26, 38, 30, 29, 2, 37, 34, 37, 50, 2, 31, 40, 46, 39, 29, 2, 26, 2, 39, 30, 30, 29, 37, 30, 2, 34, 39, 2, 33, 30, 43, 2, 43, 40, 40, 38, 10, 2, 44, 33, 30, 2, 36, 39, 30, 48, 2, 34, 45, 2, 48, 26, 44, 2, 29, 34, 31, 31, 34, 28, 46, 37, 45, 2, 45, 40, 2, 41, 37, 26, 50, 2, 48, 34, 45, 33, 2, 34, 45, 2, 27, 30, 28, 26, 46, 44, 30, 2, 34, 45, 2, 48, 26, 44, 2, 44, 33, 26, 43, 41, 10, 2, 37, 34, 37, 50, 2, 48, 26, 39, 45, 30, 29, 2, 45, 40, 2, 44, 33, 26, 43, 30, 2, 45, 33, 30, 2, 39, 30, 30, 29, 37, 30, 2, 48, 34, 45, 33, 2, 33, 30, 43, 2, 38, 40, 38, 8, 2, 44, 40, 2, 44, 33, 30, 2, 28, 40, 46, 37, 29, 2, 44, 30, 48, 2, 26, 2, 27, 46, 45, 45, 40, 39, 2, 40, 39, 2, 33, 30, 43, 2, 44, 33, 34, 43, 45, 10, 1, 37, 34, 37, 50, 2, 48, 30, 39, 45, 2, 45, 40, 2, 33, 30, 43, 2, 38, 40, 38, 2, 26, 39, 29, 2, 44, 26, 34, 29, 8, 2, 4, 38, 40, 38, 8, 2, 34, 2, 31, 40, 46, 39, 29, 2, 45, 33, 34, 44, 2, 39, 30, 30,

In [24]:
# Now let's encode the entire dataset of stories into list of lists and store them aas tensor
data = torch.tensor(encode(text_clean), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])  # The input to our model will look something like this
assert decode(encode(text_clean)) == text_clean

torch.Size([87936505]) torch.int64
tensor([40, 39, 30,  2, 29, 26, 50,  8,  2, 26,  2, 37, 34, 45, 45, 37, 30,  2,
        32, 34, 43, 37,  2, 39, 26, 38, 30, 29,  2, 37, 34, 37, 50,  2, 31, 40,
        46, 39, 29,  2, 26,  2, 39, 30, 30, 29, 37, 30,  2, 34, 39,  2, 33, 30,
        43,  2, 43, 40, 40, 38, 10,  2, 44, 33, 30,  2, 36, 39, 30, 48,  2, 34,
        45,  2, 48, 26, 44,  2, 29, 34, 31, 31, 34, 28, 46, 37, 45,  2, 45, 40,
         2, 41, 37, 26, 50,  2, 48, 34, 45, 33,  2, 34, 45,  2, 27, 30, 28, 26,
        46, 44, 30,  2, 34, 45,  2, 48, 26, 44,  2, 44, 33, 26, 43, 41, 10,  2,
        37, 34, 37, 50,  2, 48, 26, 39, 45, 30, 29,  2, 45, 40,  2, 44, 33, 26,
        43, 30,  2, 45, 33, 30,  2, 39, 30, 30, 29, 37, 30,  2, 48, 34, 45, 33,
         2, 33, 30, 43,  2, 38, 40, 38,  8,  2, 44, 40,  2, 44, 33, 30,  2, 28,
        40, 46, 37, 29,  2, 44, 30, 48,  2, 26,  2, 27, 46, 45, 45, 40, 39,  2,
        40, 39,  2, 33, 30, 43,  2, 44, 33, 34, 43, 45, 10,  1, 37, 34, 37, 50,
     

In [25]:
#  Let's now split up the data into train and validation sets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [26]:
block_size = 8 # Context window length (in number of tokens === here, number of characters)
train_data[:block_size+1], decode(train_data[:block_size+1].tolist())

(tensor([40, 39, 30,  2, 29, 26, 50,  8,  2]), 'one day, ')

In [27]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, the target is: {target}")
    print(f"Context: {decode(context.tolist())} Target: {decode([target.tolist()])}\n\n")

When input is tensor([40]), the target is: 39
Context: o Target: n


When input is tensor([40, 39]), the target is: 30
Context: on Target: e


When input is tensor([40, 39, 30]), the target is: 2
Context: one Target:  


When input is tensor([40, 39, 30,  2]), the target is: 29
Context: one  Target: d


When input is tensor([40, 39, 30,  2, 29]), the target is: 26
Context: one d Target: a


When input is tensor([40, 39, 30,  2, 29, 26]), the target is: 50
Context: one da Target: y


When input is tensor([40, 39, 30,  2, 29, 26, 50]), the target is: 8
Context: one day Target: ,


When input is tensor([40, 39, 30,  2, 29, 26, 50,  8]), the target is: 2
Context: one day, Target:  




In [28]:
torch.manual_seed(1337)
batch_size = 4  # Number of independent sequences that will be processed in parallel
context_window_size = 8 # Maximum number of tokens considered for prediction

def get_batch(split: Literal["train", "val"]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Generate a small batch of data of inputs x and targets y
    split = split.lower()
    if split not in ("train", "val"):
        raise ValueError("split must be either train or val")
    
    data = train_data if split == "train" else val_data
    ix = torch.randint(low=0, high=len(data)-context_window_size, size=(batch_size, ))
    x = torch.stack([data[i:i+context_window_size] for i in ix])
    y = torch.stack([data[i+1:i+context_window_size+1] for i in ix])
    return (ix, x, y)

In [29]:
(ix, xb, yb) = get_batch("Train")
print("Shape of ix (indices of selected tokens): ", ix.shape)
print("Shape of xb (input variables): ", xb.shape)
print("Shape of yb (target variables): ", yb.shape)

Shape of ix (indices of selected tokens):  torch.Size([4])
Shape of xb (input variables):  torch.Size([4, 8])
Shape of yb (target variables):  torch.Size([4, 8])


In [30]:
print(xb)   # our input to the transformer
print(yb)   # what we're expecting from the transformer

tensor([[30, 43, 30, 10,  1, 48, 33, 30],
        [26, 34, 29, 10,  1, 45, 33, 30],
        [27, 30, 28, 26, 46, 44, 30,  2],
        [40, 38, 34, 44, 30, 29,  2, 45]])
tensor([[43, 30, 10,  1, 48, 33, 30, 39],
        [34, 29, 10,  1, 45, 33, 30, 50],
        [30, 28, 26, 46, 44, 30,  2, 33],
        [38, 34, 44, 30, 29,  2, 45, 40]])


In [31]:
def view_data(x: torch.Tensor, y: torch.Tensor) -> None:
    for i in range(x.shape[0]): # batch dimension
        for j in range(x.shape[1]): # time dimension
            context = x[i][:j+1]
            target = y[i][j] # since y is shifted to the right by 1
            print(f"When input is: {context.tolist()} the target is: {target}")
            print(f"Context: {decode(context.tolist())} Target: {decode([target.tolist()])}")
        print("----------x----------")
view_data(xb, yb)

When input is: [30] the target is: 43
Context: e Target: r
When input is: [30, 43] the target is: 30
Context: er Target: e
When input is: [30, 43, 30] the target is: 10
Context: ere Target: .
When input is: [30, 43, 30, 10] the target is: 1
Context: ere. Target: 

When input is: [30, 43, 30, 10, 1] the target is: 48
Context: ere.
 Target: w
When input is: [30, 43, 30, 10, 1, 48] the target is: 33
Context: ere.
w Target: h
When input is: [30, 43, 30, 10, 1, 48, 33] the target is: 30
Context: ere.
wh Target: e
When input is: [30, 43, 30, 10, 1, 48, 33, 30] the target is: 39
Context: ere.
whe Target: n
----------x----------
When input is: [26] the target is: 34
Context: a Target: i
When input is: [26, 34] the target is: 29
Context: ai Target: d
When input is: [26, 34, 29] the target is: 10
Context: aid Target: .
When input is: [26, 34, 29, 10] the target is: 1
Context: aid. Target: 

When input is: [26, 34, 29, 10, 1] the target is: 45
Context: aid.
 Target: t
When input is: [26, 34, 29, 

In [32]:
torch.manual_seed(1337)
from torch.nn import Embedding
from torch.nn.functional import cross_entropy

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = Embedding(vocab_size, vocab_size)   # (num_embeddings, embed_size)
        # Each token directly reads off the logits for the next token from a lookup table
        # For this model, embed_size == vocab_size
    
    def forward(self, idx: torch.Tensor, targets=None):
        # idx and targets -> [batch_size, context_window_size)] 
        logits = self.token_embedding_table(idx)
        # logits -> [batch_size, context_window_size, embed_size]
        
        # print('-------- Original Logits and targets---------')
        # print(logits.shape)
        # print(targets.shape)
        # print(logits)
        # print(targets)
        
        if targets is None:
            loss = None
        else:
            batch, context, embed = logits.shape
            
            # print('-------- Transformed Logits and targets---------')
            # print(logits.view(batch*context, embed).shape)
            # print(targets.view(batch*context).shape)
            # print(logits.view(batch*context, embed))
            # print(targets.view(batch*context))
            
            logits = logits.view(batch*context, embed)
            # logits -> [batch_size*context_window_size, embed_size]
            targets = targets.view(batch*context)
            # targets -> [batch_size*context_window_size] 
            loss = cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx -> [batch_size, context_window_size)]
        for _ in range(max_new_tokens):
            logits, loss = self.forward(idx)
            # Todo - make the generation stop when <eos> is encountered
            
            # logits -> [batch_size, context_window_size, embed_size] 
            # Since targets is None, logits' shape stays 3 dimensional
            # loss -> [] === []  (loss is just a single floating point number)
            logits = logits[:, -1, :]   # Focus only on the previous token (not context_window, only the last time step)
            # logits -> [batch_size, embed_size] 
            probs = F.softmax(logits, dim = 1)
            # probs -> [batch_size, embed_size]
            idx_next = torch.multinomial(probs, num_samples=1) 
            # Rather than just taking the most probable one, sampling from multinomial distribution
            # idx_next -> [batch_size, 1]
            idx = torch.cat((idx, idx_next), dim=1) 
            # idx -> [batch_size, context_window + 1]
        return idx

In [33]:
mod = BigramLanguageModel(vocab_size=len(vocab))
mod

BigramLanguageModel(
  (token_embedding_table): Embedding(52, 52)
)

In [34]:
logits, loss = mod(xb, yb)

In [35]:
print(logits)
print(loss)
print(logits.shape, loss.shape)

tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.4032,  0.3027, -0.8034],
        [-1.0993,  1.3919,  0.9944,  ...,  2.3408, -2.5665,  0.9935],
        [-1.5101, -0.0948,  1.0927,  ..., -0.4032,  0.3027, -0.8034],
        ...,
        [ 1.2861,  1.0044,  0.0237,  ..., -1.2814, -0.4783,  0.2979],
        [-0.1772, -0.1334,  0.2940,  ...,  0.1700,  1.2249, -0.2345],
        [-0.4650, -0.4038,  0.5175,  ..., -0.2059,  1.3214,  0.6661]],
       grad_fn=<ViewBackward0>)
tensor(4.3525, grad_fn=<NllLossBackward0>)
torch.Size([32, 52]) torch.Size([])


In [36]:
bmodel = BigramLanguageModel(vocab_size=len(vocab))
print(bmodel)
logits, loss = bmodel(xb, yb)
print(logits.shape)
print(loss)

print(decode(bmodel.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

BigramLanguageModel(
  (token_embedding_table): Embedding(52, 52)
)
torch.Size([32, 52])
tensor(4.2026, grad_fn=<NllLossBackward0>)
	?:z;u.gmba/l85f-9'ij
m(
h:(<|endoftext|>o'l v!7	'-sl?
0t	6uhvp0 	26,;fj,	/ytsl(5g0,6".c0l a0zo9"!6o ?'yzc'g/<|endoftext|>?lp


In [37]:
optimizer = torch.optim.AdamW(bmodel.parameters(), lr=1e-3)
batch_size = 32

for steps in range(1000):
    _, xb, yb = get_batch('train')
    logits, loss = bmodel(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

3.245846748352051


In [38]:
print(decode(bmodel.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

	b	7!xuc;',!ee37e3:n,;vya;zy;,mcqlarmv"cr9c'0-4un8m n8q1meler;ay	'!0mi4/''de0(q(<|endoftext|>z
6okg564g4'0-veh5)v


In [ ]:
## Self Attention

torch.manual_seed(1337)
# input -> [batch_size, context_window_size, embed_size]
batch_size, context_window_size, embed_size = 4, 8, 32
xsa = torch.randn(batch_size, context_window_size, embed_size)
print(f"Input shape looks like: {xsa.shape}\n")
# print("Input looks like: ", xsa)

head_size = 16
qw = nn.Linear(embed_size, head_size, bias=False)
kw = nn.Linear(embed_size, head_size, bias=False)
vw = nn.Linear(embed_size, head_size, bias=False)

q = qw(xsa)   # q -> [batch_size, context_window_size, head_size]
k = kw(xsa)   # k -> [batch_size, context_window_size, head_size]
print(f"Query matrix: {q.shape}, Key matrix: {k.shape}\n")

wei = q @ k.transpose(-2, -1)   # wei -> [batch_size, context_window_size, context_window_size]
print(f"Dot product of Query and Key matrices: {wei.shape}\n")
mask = torch.tril((torch.ones(context_window_size, context_window_size)), diagonal=0)   # mask -> [context_window_size, context_window_size]
print("Mask for hiding future words at a given timestep:\n", mask)

print(f"Single batch of weight matrix:\n{wei[0]}")
wei = wei.masked_fill(mask == 0, float('-inf')) # wei -> [batch_size, context_window_size, context_window_size]
print(f"Single batch of masked weight matrix:\n{wei[0]}")

wei = wei / (head_size**0.5)
wei = F.softmax(wei, dim=-1) # wei -> [batch_size, context_window_size, context_window_size]
print(f"Single batch of updated attention weight matrix:\n{wei[0]}")

v = vw(xsa) # v -> [batch_size, context_window_size, head_size]
out = wei @ v
print(f"Output matrix: {out.shape}")

Input shape looks like: torch.Size([4, 8, 32])

Query matrix: torch.Size([4, 8, 16]), Key matrix: torch.Size([4, 8, 16])

Dot product of Query and Key matrices: torch.Size([4, 8, 8])

Mask for hiding future words at a given timestep:
 tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
Single batch of weight matrix:
tensor([[-1.7629, -3.3334, -1.0226,  0.7836, -1.2566, -0.3126,  1.0876, -1.8044],
        [-1.3011, -1.6556, -1.2606, -0.8014,  0.0187,  2.4152,  1.9652, -0.4126],
        [ 0.5652,  0.1040,  0.0762, -0.3368, -0.7880, -0.1106, -0.2621, -0.8306],
        [ 2.1616,  3.3782, -0.3813, -0.8496, -1.3204, -0.9931, -0.3158,  0.5899],
        [-1.0674, -2.1825, -0.9843, -0.5602,  2.0363,  3.3449,  0.609

In [3]:
from datetime import datetime

datetime.now().strftime("%d-%m-%y-%H-%M-%S") + "_bigram"

'13-04-26-23-27-40_bigram'